# Scaling to Production

> Validate cluster and TLS configs with LMCache's own parser, monitor cache hits from Valkey, and wire the connector under a real vLLM server.

**Advanced** · Python · ~20 min

**Who is this for:** Developers taking LMCache + Valkey from a laptop demo to a real deployment — cluster-mode Valkey, TLS, capacity planning, monitoring, and the full vLLM inference path.

## What changes in production

The earlier notebooks ran LMCache's connector directly on CPU so you could see the mechanics. In production the same connector is driven by a vLLM server, and Valkey is typically clustered and TLS-protected. This notebook covers the Valkey-facing parts you configure and operate — and every config here is validated by loading it through LMCache's **real** parser, so a typo fails on your laptop instead of on a GPU host.

## Prerequisites

- Completed [01](01-intro-to-kv-caching-with-valkey.ipynb) and [02](02-sharing-kv-caches-across-instances.ipynb)
- Docker, and the dependencies from `requirements.txt`
- A local Valkey for the monitoring section

## Step 1: Start Valkey

```bash
docker compose up -d --wait
```

> ⚠️ **Security:** These examples use no authentication or TLS for simplicity.
> For any non-localhost deployment, enable authentication and TLS.
> See the [Valkey security documentation](https://valkey.io/topics/security/).

In [ ]:
import subprocess
print(subprocess.run(["docker", "exec", "valkey", "valkey-cli", "ping"],
                     capture_output=True, text=True).stdout.strip())

## Step 2: Validate a cluster-mode config

Cluster mode distributes KV chunks across shards. You enable it in the LMCache
config via `extra_config.valkey_mode: "cluster"`; the GLIDE client
auto-discovers the topology from a seed node. We validate the config by loading
it through LMCache's real parser — if it raises here, it would raise on the GPU
host too.

In [ ]:
import tempfile
from common import load_lmcache_config

cluster_yaml = """chunk_size: 256
local_cpu: true
max_local_cpu_size: 10.0
remote_url: "valkey://localhost:6379"
remote_serde: "cachegen"
extra_config:
  valkey_mode: "cluster"
  valkey_num_workers: 32
"""

path = tempfile.mktemp(suffix=".yaml")
with open(path, "w") as f:
    f.write(cluster_yaml)

config = load_lmcache_config(path)
print("valkey_mode:", config.extra_config["valkey_mode"])
print("num_workers:", config.extra_config["valkey_num_workers"])
print("serde:", config.remote_serde)

For a TLS-enabled endpoint, add `tls_enable: true` under `extra_config` and
use a `valkeys://` URL. Inject credentials (`valkey_username`/`valkey_password`)
at deploy time via a secrets manager — never hardcode them in a config file
checked into source control. For managed Valkey services, consult your
provider's documentation for the exact TLS and auth settings.

## Step 3: Validate an MP-mode adapter config

LMCache's newer **MP mode** runs LMCache as a standalone server that vLLM
connects to over ZMQ, with Valkey as its L2 adapter. Its config is a JSON
object passed to `--l2-adapter`, validated by a different real parser,
`ValkeyL2AdapterConfig.from_dict`:

In [ ]:
from lmcache.v1.distributed.l2_adapters.valkey_l2_adapter import ValkeyL2AdapterConfig

adapter = ValkeyL2AdapterConfig.from_dict({
    "type": "valkey",
    "startup_nodes": "localhost:6379",
    "cluster_mode": True,
    "num_workers": 16,
})
print("MP adapter parsed OK:", type(adapter).__name__)

> **Note:** MP-mode Valkey support is newer than the in-process `valkey://`
> connector used in notebooks 01–02. The in-process connector is what most
> existing LMCache + Valkey deployments run today; MP mode is the direction the
> project is heading. Both drive the same Valkey storage.

## Step 4: Monitor cache hits from Valkey

Valkey's own stats tell you whether the cache is earning its keep. `keyspace_hits`
vs `keyspace_misses` is the hit rate; `used_memory` tracks footprint.

In [ ]:
out = subprocess.run(
    ["docker", "exec", "valkey", "valkey-cli", "INFO", "stats"],
    capture_output=True, text=True,
).stdout
for line in out.splitlines():
    if line.startswith(("keyspace_hits", "keyspace_misses")):
        print(line.strip())

In [ ]:
mem = subprocess.run(
    ["docker", "exec", "valkey", "valkey-cli", "INFO", "memory"],
    capture_output=True, text=True,
).stdout
for line in mem.splitlines():
    if line.startswith("used_memory_human"):
        print(line.strip())

## Capacity planning and eviction

Estimate Valkey memory from the number of unique prompt prefixes you expect to
cache; each LMCache chunk holds `chunk_size` tokens of KV data. See
[LMCache's documentation](https://docs.lmcache.ai) for per-model KV size
estimates.

For an evictable remote cache, cap memory and let Valkey evict least-recently-used
entries under pressure:

```conf
maxmemory <limit>
maxmemory-policy allkeys-lru
```

Then watch `used_memory` and `evicted_keys` (via `INFO memory` / `INFO stats`)
and tune `maxmemory` from observed peak usage. See Valkey's
[key eviction documentation](https://valkey.io/topics/lru-cache/).

## Running the full vLLM inference path

Everything above drives LMCache's Valkey connector directly on CPU. The
production step is to let a **vLLM** server drive that same connector from its
KV cache, using vLLM's `LMCacheConnectorV1`:

```python
from vllm import LLM
from vllm.config import KVTransferConfig

llm = LLM(
    model="facebook/opt-125m",
    kv_transfer_config=KVTransferConfig(
        kv_connector="LMCacheConnectorV1",
        kv_role="kv_both",
    ),
)
```

with LMCache pointed at Valkey via `LMCACHE_REMOTE_URL=valkey://<host>:6379`.

**Practical requirements for this path:**

- **Run it on Linux x86-64.** vLLM publishes no CPU wheel on PyPI; use the
  official prebuilt CPU image
  (`public.ecr.aws/q9t5s3a7/vllm-cpu-release-repo`) or
  [build vLLM from source](https://docs.vllm.ai/en/stable/getting_started/installation/cpu.html).
  The prebuilt CPU image is compiled with **AVX512**, so it needs a host CPU
  that supports it — it raises `Illegal instruction` (SIGILL) on CPUs without
  AVX512, including x86 emulated on Apple Silicon.
- **macOS/ARM** support in vLLM is experimental and requires building from
  source.
- **A GPU is only needed for meaningful time-to-first-token numbers.** The
  correctness of the LMCache↔Valkey path (store, share, and load KV) is exactly
  what notebooks 01–02 demonstrate on CPU; the GPU speeds up the model's own
  compute, not the caching.

Because this path needs a specific platform and (for real latency numbers) a
GPU, it's an optional extension — the runnable default in this cookbook is the
CPU connector path.

## Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `ConnectionError` on startup | Valkey unreachable | Check `remote_url`/`startup_nodes`, network, and TLS settings |
| 0% cache hits across workers | Different `PYTHONHASHSEED` per worker | Export `PYTHONHASHSEED=0` on all workers |
| Config fails to load | Invalid YAML or unknown key | Load it through LMCache's real parser locally first (Steps 2–3) |
| Valkey out of memory | Too many cached chunks | Set `maxmemory` with `allkeys-lru`, or add cluster shards |
| `Illegal instruction` running vLLM CPU | Prebuilt image needs AVX512 | Use an AVX512 host or build vLLM from source |

## Teardown

```bash
docker compose down
```

---

[← 02 - Sharing KV Caches Across Instances](02-sharing-kv-caches-across-instances.ipynb) | [README →](README.md)